# 1.1.6 Functions

Decorators, generators, closures, partial functions — applied to ML utility patterns.

In [ ]:
import functools, time, math, random

# Timer decorator
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        print(f'{func.__name__} took {(time.perf_counter()-t0)*1000:.2f} ms')
        return result
    return wrapper

@timer
def compute_stats(data):
    mean = sum(data)/len(data)
    std  = math.sqrt(sum((x-mean)**2 for x in data)/len(data))
    return mean, std

random.seed(0)
data = [random.gauss(0,1) for _ in range(100_000)]
mu, s = compute_stats(data)
print(f'mean={mu:.4f}, std={s:.4f}')

In [ ]:
# Memoization with functools.lru_cache
@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

results = {n: fib(n) for n in range(0, 15)}
print('Fibonacci:', results)
print('Cache info:', fib.cache_info())

In [ ]:
# Generators for streaming
def infinite_batch(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i+batch_size]

data = list(range(1000))
batch_means = [sum(b)/len(b) for b in infinite_batch(data, 50)]
print(f'Batch means (first 5): {batch_means[:5]}')
print(f'Total batches: {len(batch_means)}')

In [ ]:
# Closures: running statistics tracker
def make_tracker():
    history = []
    def track(val):
        history.append(val)
        return {'last': val, 'mean': sum(history)/len(history),
                'best': min(history), 'n': len(history)}
    return track

tracker = make_tracker()
random.seed(42)
for i in range(1, 8):
    loss = 1.0 * math.exp(-i*0.3) + random.gauss(0, 0.02)
    s = tracker(loss)
    print(f'epoch={i:2d}  loss={s["last"]:.4f}  mean={s["mean"]:.4f}  best={s["best"]:.4f}')

In [ ]:
# Partial functions
normalize = lambda x, lo, hi: (x - lo) / (hi - lo + 1e-8)
norm_fare  = functools.partial(normalize, lo=0.0, hi=512.0)
norm_age   = functools.partial(normalize, lo=0.0, hi=80.0)

print('Fare normalization:', [round(norm_fare(f),3) for f in [0, 50, 100, 512]])
print('Age  normalization:', [round(norm_age(a), 3) for a in [0, 20, 40, 80]])